# 1. Basic graph classes

The `Node` and `Graph` classes describe the graphs used in this project.
A vertex represents an individual, and an edge records a connection between
two individuals. The graphs are **simple and undirected**: a vertex cannot
connect to itself, a pair of vertices has at most one edge, and a connection
works in both directions.

Use the project's `.venv` kernel. The next notebook, `02_algorithms.ipynb`,
loads these classes automatically with `%run`. Experiments and saved results
come later in the simulation and analysis notebooks.

In [ ]:
import uuid
from numbers import Integral

## A vertex: identity, neighbours, and degree

A `Node` represents one vertex. Its `neighbours` are the vertices connected
to it, and its `degree` is the number of those neighbours. The degree is
stored and updated whenever a connection changes, so algorithms can read it
without counting the neighbours again.

Each vertex also has a unique label called a UUID. Comparisons between nodes
use this label, so a vertex can still be recognised in a copy of a graph.
The labels have no role in the probability model.

`join()` and `unjoin()` change a connection for both vertices. If the
vertices belong to a graph, they also update its record of edges. The
`Graph.add_edge()` and `Graph.remove_edge()` methods additionally check
that the supplied vertices belong to that graph. Invalid or repeated operations
raise `ValueError`, so a mistake stops the calculation where it occurs.
Passing something other than a `Node` raises `TypeError`. Use these methods
rather than editing `neighbours`, `degree`, or `id` directly, so the stored
information stays consistent.

In [ ]:
class Node:
    """A vertex with a unique label and a set of neighbouring vertices.

    The stored degree is the number of neighbours. Adding or removing
    a connection updates both vertices and, when present, their graph.
    """

    def __init__(self):
        self.id = uuid.uuid4()
        self.neighbours = set()
        self.degree = 0
        # A vertex can belong to one graph at a time. None means standalone.
        self._graph = None

    def join(self, node):
        """Connect two vertices and update their graph if they have one."""
        if not isinstance(node, Node):
            raise TypeError("node must be a Node")
        if node == self:
            raise ValueError("A vertex cannot be joined to itself")
        if self._graph is not node._graph:
            raise ValueError(
                "Vertices must belong to the same graph or both be standalone"
            )
        if node in self.neighbours or self in node.neighbours:
            raise ValueError("These vertices are already joined")
        edge = frozenset({self, node})
        if self._graph is not None and edge in self._graph.edges:
            raise ValueError("This edge is already in the graph")

        self.neighbours.add(node)
        node.neighbours.add(self)
        self.degree += 1
        node.degree += 1
        if self._graph is not None:
            self._graph.edges.add(edge)

    def unjoin(self, node):
        """Disconnect two neighbours and update their graph if present."""
        if not isinstance(node, Node):
            raise TypeError("node must be a Node")
        if self._graph is not node._graph:
            raise ValueError(
                "Vertices must belong to the same graph or both be standalone"
            )
        if node not in self.neighbours or self not in node.neighbours:
            raise ValueError("These vertices are not joined")
        edge = frozenset({self, node})
        if self._graph is not None and edge not in self._graph.edges:
            raise ValueError("This edge is not in the graph")

        self.neighbours.remove(node)
        node.neighbours.remove(self)
        self.degree -= 1
        node.degree -= 1
        if self._graph is not None:
            self._graph.edges.remove(edge)

    def __eq__(self, other):
        if not isinstance(other, Node):
            return NotImplemented
        return self.id == other.id

    def __hash__(self):
        return hash(self.id)

## A graph: vertices and unordered edges

`Graph(n)` starts with $n$ vertices and no edges. The `nodes` set holds the
vertices, and the `edges` set holds the connections between them. Each edge
is stored as a `frozenset` containing its two vertices: their order does not
matter. Removing a vertex also removes every edge attached to it.

Add only isolated vertices with `add_node()`, then connect them with
`add_edge()`. A vertex belongs to one graph at a time. The private `_graph`
reference checks this, including when a copied vertex has the same UUID.
Use vertices from the particular graph you want to change. A removed
vertex becomes standalone and can be added to another graph. Invalid
operations raise an exception before changing the graph.

Core pruning works on a copy so we can compare the input graph with its core.
The `__deepcopy__()` method creates separate vertices and connections while
keeping their UUID labels. Changes to either graph do not affect the other.

Generated random intersection graphs also keep a record of how they were made:

- `memberships` records how many community memberships were assigned to each
  vertex, including vertices with no memberships.
- `communities` records the vertex label assigned to each place in every
  community. These records are tuples, so they cannot be edited in place.
  One vertex can occupy more than one place in a community.

These records stay unchanged if vertices are removed or a core is taken.
Use `nodes`, `edges`, and each node's `degree` to inspect the graph as it is
now. A graph built by hand starts with empty membership records. The next
notebook explains how memberships are sampled and turned into edges.

In [ ]:
class Graph:
    """An undirected graph that also keeps its sampled membership records."""

    def __init__(self, n=0):
        if isinstance(n, bool) or not isinstance(n, Integral) or n < 0:
            raise ValueError("n must be a nonnegative integer")
        self.nodes = set()
        self.edges = set()
        # These records are filled when a random intersection graph is made.
        self.memberships = {}
        self.communities = ()
        for _ in range(n):
            self.add_node(Node())

    def add_node(self, node):
        """Add an isolated vertex that does not already belong to a graph."""
        if not isinstance(node, Node):
            raise TypeError("node must be a Node")
        if node in self.nodes:
            raise ValueError("This vertex is already in the graph")
        if node._graph is not None:
            raise ValueError("This vertex already belongs to another graph")
        if node.neighbours or node.degree != 0:
            raise ValueError("Only isolated vertices can be added to a graph")
        self.nodes.add(node)
        node._graph = self

    def _check_node(self, node):
        """Require a vertex from this graph, rather than an equal copy."""
        if not isinstance(node, Node):
            raise TypeError("node must be a Node")
        if node._graph is not self or node not in self.nodes:
            raise ValueError("This vertex does not belong to the graph")

    def remove_node(self, node):
        """Remove a vertex and every edge attached to it."""
        self._check_node(node)
        # Use a copy because removing edges changes the neighbour set.
        for neighbour in node.neighbours.copy():
            self.remove_edge(node, neighbour)
        self.nodes.remove(node)
        node._graph = None

    def add_edge(self, node1, node2):
        """Add an edge between two vertices already belonging to the graph."""
        self._check_node(node1)
        self._check_node(node2)
        if node1 == node2:
            raise ValueError("A vertex cannot be joined to itself")
        edge = frozenset({node1, node2})
        if edge in self.edges:
            raise ValueError("This edge is already in the graph")
        node1.join(node2)

    def remove_edge(self, node1, node2):
        """Remove an edge and update both endpoints' neighbour sets."""
        self._check_node(node1)
        self._check_node(node2)
        edge = frozenset({node1, node2})
        if edge not in self.edges:
            raise ValueError("This edge is not in the graph")
        node1.unjoin(node2)

    def __deepcopy__(self, memo):
        """Make an independent graph copy, keeping the same vertex labels."""
        new_graph = Graph()
        memo[id(self)] = new_graph

        # Link each input vertex to a separate copy with the same label.
        node_map = {}
        for node in self.nodes:
            new_node = Node()
            new_node.id = node.id
            node_map[node] = new_node
            memo[id(node)] = new_node
            new_graph.add_node(new_node)

        # Rebuild the edges so the two graphs have separate neighbour sets.
        for edge in self.edges:
            node1, node2 = tuple(edge)
            new_graph.add_edge(node_map[node1], node_map[node2])

        # Copy the counts; the community tuples can safely be shared.
        # Keep this sampling history even if vertices are later removed.
        new_graph.memberships = self.memberships.copy()
        new_graph.communities = self.communities
        return new_graph